# IC50 Plate Layout Generator
Run cells 1 → 2 → 3 in order.

**Output:** `plate_layout_<ID>_<date>.xlsx` with two sheets:
- **Setup** — experiment metadata, concentration series, conditions table (fill in Cell Line Lot + Compound Lot after downloading)
- **Plate Maps** — visual 96-well layout per plate for bench use


In [1]:
# @title Cell 1 — Install & import
!pip install openpyxl --quiet
import ipywidgets as widgets
from IPython.display import display, HTML
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import datetime
from google.colab import files
print("Ready.")


Ready.


In [2]:
# @title Cell 2 — Experiment configuration
style  = {'description_width': '200px'}
layout = widgets.Layout(width='440px')
lshort = widgets.Layout(width='320px')

display(HTML("<h3 style='font-family:Arial;color:#1F6B75'>Experiment Details</h3>"))
w_exp_id   = widgets.Text(description='Experiment ID:',       style=style, layout=layout)
w_date     = widgets.Text(description='Date (YYYY-MM-DD):',
                          value=str(datetime.date.today()),   style=style, layout=layout)
w_operator = widgets.Text(description='Operator:',            style=style, layout=layout)
w_notes    = widgets.Textarea(description='Notes:',           style=style,
                               layout=widgets.Layout(width='440px', height='60px'))
w_n_plates = widgets.IntText(description='Number of plates:', value=1,
                              style=style, layout=lshort)
display(w_exp_id, w_date, w_operator, w_notes, w_n_plates)

display(HTML("<h3 style='font-family:Arial;color:#1F6B75;margin-top:16px'>Concentration Series</h3>"))
w_high = widgets.FloatText(description='Highest dose (µM):', value=10.0, style=style, layout=lshort)
w_dil  = widgets.FloatText(description='Dilution factor:',   value=3.16, style=style, layout=lshort)
w_mode = widgets.RadioButtons(
    options=['Auto-calculate', 'Manual entry'],
    value='Auto-calculate', description='Mode:',
    style=style, layout=widgets.Layout(width='400px'))

manual_box    = widgets.VBox([])
manual_inputs = []

def build_manual():
    global manual_inputs
    manual_inputs = [
        widgets.FloatText(description=f'Dose {i+1} (µM):',
                          value=round(10.0 / (3.16**i), 4),
                          style={'description_width':'120px'},
                          layout=widgets.Layout(width='300px'))
        for i in range(9)
    ]
    manual_box.children = manual_inputs

def on_mode(change):
    manual_box.children = [] if change['new'] == 'Auto-calculate' else (build_manual() or manual_inputs)

w_mode.observe(on_mode, names='value')
display(w_high, w_dil, w_mode, manual_box)
print("\nFill in the fields above, then run Cell 3.")


Text(value='', description='Experiment ID:', layout=Layout(width='440px'), style=DescriptionStyle(description_…

Text(value='2026-06-24', description='Date (YYYY-MM-DD):', layout=Layout(width='440px'), style=DescriptionStyl…

Text(value='', description='Operator:', layout=Layout(width='440px'), style=DescriptionStyle(description_width…

Textarea(value='', description='Notes:', layout=Layout(height='60px', width='440px'), style=DescriptionStyle(d…

IntText(value=1, description='Number of plates:', layout=Layout(width='320px'), style=DescriptionStyle(descrip…

FloatText(value=10.0, description='Highest dose (µM):', layout=Layout(width='320px'), style=DescriptionStyle(d…

FloatText(value=3.16, description='Dilution factor:', layout=Layout(width='320px'), style=DescriptionStyle(des…

RadioButtons(description='Mode:', layout=Layout(width='400px'), options=('Auto-calculate', 'Manual entry'), st…

VBox()


Fill in the fields above, then run Cell 3.


In [3]:
# @title Cell 3 — Generate plate_layout.xlsx

# ── Palette (minimal) ─────────────────────────────────────────
HDR    = "4A7C88"   # dark teal  — section headers (white text)
SUBHDR = "D0D0D0"   # light gray — column headers  (black text)
WHITE  = "FFFFFF"
BLACK  = "000000"
CTRL   = "C6EFCE"   # light green — control wells
EDGE   = "E8E8E8"   # light gray  — media/edge wells

# Subtle pastels for condition rows — black text always readable
COND_PALETTE = [
    "F0E6FF","E6F0FF","FFF6E6","E6FFE6","FFE6E6","E6E6FF",
    "FFFFF0","F0FFFF","FFE6F0","E6F0FF","FFF0E6","F0FFE6",
    "F5E6FF","E6FFF5","F5FFE6","E6F5FF","FFE6F5","F5FFE6",
]

def xfill(h):
    return PatternFill("solid", start_color=h, fgColor=h)

def xfont(bold=False, color=BLACK, sz=10, italic=False):
    return Font(name="Arial", bold=bold, color=color, size=sz, italic=italic)

def xborder():
    s = Side(style="thin")
    return Border(left=s, right=s, top=s, bottom=s)

def ctr(): return Alignment(horizontal="center", vertical="center", wrap_text=True)
def lft(): return Alignment(horizontal="left",   vertical="center", wrap_text=True)

def hdr_cell(ws, ref, text, merge_to=None):
    if merge_to:
        ws.merge_cells(f"{ref}:{merge_to}")
    ws[ref] = text
    ws[ref].font      = Font(name="Arial", bold=True, color=WHITE, size=10)
    ws[ref].fill      = xfill(HDR)
    ws[ref].alignment = lft()
    ws[ref].border    = xborder()

def col_hdr(c, text):
    c.value     = text
    c.font      = xfont(bold=True)
    c.fill      = xfill(SUBHDR)
    c.alignment = ctr()
    c.border    = xborder()

def plain(c, value=None, bold=False, italic=False, al=None):
    if value is not None: c.value = value
    c.font      = xfont(bold=bold, italic=italic)
    c.fill      = xfill(WHITE)
    c.alignment = al or ctr()
    c.border    = xborder()

# ── Read widgets ──────────────────────────────────────────────
n_plates = max(1, w_n_plates.value)
exp_id   = w_exp_id.value   or "Experiment"
date_str = w_date.value
operator = w_operator.value
notes    = w_notes.value

if w_mode.value == 'Auto-calculate':
    concs    = [round(w_high.value / (w_dil.value ** i), 6) for i in range(9)]
    mode_str = f"Auto  |  Highest: {w_high.value} µM  |  Dilution: {w_dil.value}"
else:
    concs    = [w.value for w in manual_inputs]
    mode_str = "Manual"

# Plate geometry constants
DOSE_COUNT   = 9
CTRL_COL     = 11   # plate column index for control
EDGE_COLS    = {1, 12}
EDGE_ROWS    = {"A", "H"}
INNER_ROWS   = list("BCDEFG")
ROW_PAIRS    = ["B,E", "C,F", "D,G"]
n_conds      = n_plates * 3
COND_START   = 21   # first data row in conditions table

# Build row-letter -> condition-index-within-plate lookup from ROW_PAIRS
# e.g. ROW_PAIRS = ["B,E","C,F","D,G"] -> {"B":0,"E":0,"C":1,"F":1,"D":2,"G":2}
ROW_TO_COND_IDX = {}
for ci, pair_str in enumerate(ROW_PAIRS):
    for rl in pair_str.split(","):
        ROW_TO_COND_IDX[rl.strip()] = ci

wb = Workbook()

# ════════════════════════════════════════════════════
# SETUP SHEET
# ════════════════════════════════════════════════════
ws = wb.active
ws.title = "Setup"
ws.sheet_view.showGridLines = False

ws.column_dimensions["A"].width = 10  # Plate #
ws.column_dimensions["B"].width = 16  # Cell Line
ws.column_dimensions["C"].width = 18  # Cell Line Lot
ws.column_dimensions["D"].width = 18  # Compound Name
ws.column_dimensions["E"].width = 18  # Compound Lot
ws.column_dimensions["F"].width = 18  # Other Additives
ws.column_dimensions["G"].width = 12  # Rows
ws.column_dimensions["H"].width = 10  # Cond #

last_conc_col = get_column_letter(DOSE_COUNT + 2)   # K = dose 9 header; ctrl

for r in range(1, COND_START + n_conds + 5):
    ws.row_dimensions[r].height = 18
ws.row_dimensions[1].height = 26

# Title
ws.merge_cells(f"A1:H1")
ws["A1"] = f"IC50 Plate Layout  —  {exp_id}  —  {date_str}"
ws["A1"].font      = Font(name="Arial", bold=True, color=WHITE, size=12)
ws["A1"].fill      = xfill(HDR)
ws["A1"].alignment = ctr()

# Metadata
hdr_cell(ws, "A3", "EXPERIMENT METADATA", "E3")
for r, label, value in [
    (4,  "Experiment ID",      exp_id),
    (5,  "Date (YYYY-MM-DD)",  date_str),
    (6,  "Operator",           operator),
    (7,  "Notes",              notes),
    (8,  "Number of plates",   n_plates),
]:
    ws[f"A{r}"] = label
    ws[f"A{r}"].font = xfont(); ws[f"A{r}"].alignment = lft()
    ws.merge_cells(f"B{r}:H{r}")
    plain(ws[f"B{r}"], value, al=lft())
    ws[f"B{r}"].font = xfont(bold=True)

# Concentration section
hdr_cell(ws, "A10", "GLOBAL CONCENTRATION SERIES", "H10")
ws.row_dimensions[11].height = 30
ws.merge_cells("A11:H11")
ws["A11"] = ("This is the concentration series computed globally from your inputs above. "
             "If individual plates require different concentrations, modify those plates "
             "directly on the 'Plate Maps' sheet using the per-plate calculator.")
ws["A11"].font      = xfont(italic=True, color="555555", sz=9)
ws["A11"].alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)

ws["A12"] = "Mode"; ws["A12"].font = xfont(); ws["A12"].alignment = lft()
ws.merge_cells("B12:H12")
plain(ws["B12"], mode_str, al=lft())

# Dose headers (row 14)
col_hdr(ws["A14"], "Dose #")
for i in range(DOSE_COUNT):
    col_hdr(ws[f"{get_column_letter(i+2)}14"], f"Dose {i+1}")
c = ws[f"{get_column_letter(DOSE_COUNT+2)}14"]
c.value = "Control (0 µM)"; c.font = xfont(bold=True)
c.fill = xfill(CTRL); c.alignment = ctr(); c.border = xborder()
ws.column_dimensions[get_column_letter(DOSE_COUNT+2)].width = 14

# Active concentrations (row 15)
ws["A15"] = "Concentration (µM)"; ws["A15"].font = xfont(bold=True); ws["A15"].alignment = lft()
for i, cv in enumerate(concs):
    c = ws[f"{get_column_letter(i+2)}15"]
    plain(c, round(cv, 4), bold=True)
    c.number_format = "0.0000"
c = ws[f"{get_column_letter(DOSE_COUNT+2)}15"]
c.value = 0; c.font = xfont(bold=True)
c.fill = xfill(CTRL); c.alignment = ctr(); c.border = xborder()

# Concentrations in nM (row 16)
ws["A16"] = "Concentration (nM)"; ws["A16"].font = xfont(italic=True, color="666666"); ws["A16"].alignment = lft()
for i, cv in enumerate(concs):
    c = ws[f"{get_column_letter(i+2)}16"]
    c.value = round(cv * 1000, 2)
    c.font = xfont(italic=True, color="666666")
    c.fill = xfill(WHITE); c.alignment = ctr(); c.border = xborder()
    c.number_format = "0.00"
c = ws[f"{get_column_letter(DOSE_COUNT+2)}16"]
c.value = 0; c.font = xfont(italic=True, color="666666")
c.fill = xfill(CTRL); c.alignment = ctr(); c.border = xborder()

# Conditions table
hdr_cell(ws, f"A{COND_START-2}", "CONDITIONS", f"H{COND_START-2}")
for col, h in [("A","Plate #"),("B","Cell Line"),("C","Cell Line Lot"),
               ("D","Compound Name"),("E","Compound Lot"),("F","Other Additives"),
               ("G","Rows"),("H","Cond #")]:
    col_hdr(ws[f"{col}{COND_START-1}"], h)

for i in range(n_conds):
    r     = COND_START + i
    color = COND_PALETTE[i % len(COND_PALETTE)]
    ws.row_dimensions[r].height = 18

    plain(ws[f"A{r}"], (i // 3) + 1)
    # User-fillable columns — highlighted with condition color
    for col in ["B", "C", "D", "E", "F"]:
        ws[f"{col}{r}"].fill      = xfill(color)
        ws[f"{col}{r}"].font      = xfont()
        ws[f"{col}{r}"].border    = xborder()
        ws[f"{col}{r}"].alignment = ctr()
    plain(ws[f"G{r}"], ROW_PAIRS[i % 3])
    plain(ws[f"H{r}"], i + 1, bold=True)

note_r = COND_START + n_conds
ws.merge_cells(f"A{note_r}:H{note_r}")
ws[f"A{note_r}"] = ("Fill in Cell Line, Cell Line Lot, Compound Name, Compound Lot, "
                     "and Other Additives (cols B-F). "
                     "Edit Rows (col G) if replicates are not in the default B,E / C,F / D,G pairing — "
                     "e.g. enter B,E for non-adjacent replicates. Comma-separated, rows must be B-G.")
ws[f"A{note_r}"].font = xfont(italic=True, sz=8, color="888888"); ws[f"A{note_r}"].alignment = lft()

# ════════════════════════════════════════════════════
# PLATE MAPS SHEET
# ════════════════════════════════════════════════════
pm = wb.create_sheet("Plate Maps")
pm.sheet_view.showGridLines = False

pm.column_dimensions["A"].width = 5
pm.column_dimensions["B"].width = 7        # left edge col
for i in range(DOSE_COUNT):
    pm.column_dimensions[get_column_letter(i+3)].width = 10   # dose cols
pm.column_dimensions[get_column_letter(DOSE_COUNT+3)].width = 12  # ctrl col
pm.column_dimensions[get_column_letter(DOSE_COUNT+4)].width = 7   # right edge col
pm.column_dimensions[get_column_letter(DOSE_COUNT+6)].width = 28  # legend

BLOCK_H = 13
for p in range(n_plates):
    plate_num  = p + 1
    ro         = p * BLOCK_H + 1   # row offset

    for ri in range(BLOCK_H):
        pm.row_dimensions[ro+ri].height = 20 if ri < 11 else 5

    # Title
    te = get_column_letter(DOSE_COUNT+4)
    pm.merge_cells(f"A{ro}:{te}{ro}")
    pm[f"A{ro}"] = f"Plate {plate_num}  —  {exp_id}  —  {date_str}"
    pm[f"A{ro}"].font = Font(name="Arial", bold=True, color=WHITE, size=11)
    pm[f"A{ro}"].fill = xfill(HDR); pm[f"A{ro}"].alignment = ctr()

    # Column number headers 1–12
    for pc in range(1, 13):
        col_hdr(pm[f"{get_column_letter(pc+1)}{ro+1}"], pc)

    # Wells
    for ri, rl in enumerate("ABCDEFGH"):
        er = ro + 2 + ri
        # Row label
        pm[f"A{er}"].value = rl
        pm[f"A{er}"].font  = xfont(bold=True)
        pm[f"A{er}"].fill  = xfill(SUBHDR)
        pm[f"A{er}"].alignment = ctr(); pm[f"A{er}"].border = xborder()

        for pc in range(1, 13):
            ec = get_column_letter(pc+1)
            c  = pm[f"{ec}{er}"]
            c.border = xborder(); c.alignment = ctr()

            is_edge = rl in EDGE_ROWS or pc in EDGE_COLS

            if is_edge:
                c.value = "media"
                c.fill  = xfill(EDGE)
                c.font  = xfont(sz=8, italic=True, color="888888")

            elif pc == CTRL_COL:
                if rl in ROW_TO_COND_IDX:
                    ci_local = ROW_TO_COND_IDX[rl]
                    gc = p*3 + ci_local + 1
                    c.value = f"C{gc} 0 uM"
                    c.fill  = xfill(CTRL)
                    c.font  = xfont(bold=True, sz=8)
                else:
                    c.value = "media"; c.fill = xfill(EDGE)
                    c.font  = xfont(sz=8, italic=True, color="888888")

            elif 2 <= pc <= 10:   # dose columns
                if rl in ROW_TO_COND_IDX:
                    dose_idx  = pc - 2
                    ci_local  = ROW_TO_COND_IDX[rl]
                    gc        = p*3 + ci_local + 1
                    color     = COND_PALETTE[(gc-1) % len(COND_PALETTE)]
                    conc_cell = f"{get_column_letter(pc+1)}{ro+10}"
                    c.value   = f'="C{gc} "&TEXT({conc_cell},"0.000")&" uM"'
                    c.fill    = xfill(color)
                    c.font    = xfont(sz=8)
                else:
                    c.value = "media"; c.fill = xfill(EDGE)
                    c.font  = xfont(sz=8, italic=True, color="888888")
            else:
                c.fill = xfill(WHITE); c.font = xfont(sz=8)

    # ── Per-plate concentration calculator ───────────────────
    # Calculator lives to the right of the plate block.
    # Input cells: max dose (row ro+2) and dilution factor (row ro+3).
    # Conc label row (ro+10) cells C-K reference these inputs via formulas.
    clc  = get_column_letter(DOSE_COUNT+8)   # calculator label col  (Q)
    cinp = get_column_letter(DOSE_COUNT+9)   # calculator input col  (R)

    # Calculator header
    pm[f"{clc}{ro+1}"] = "Per-plate calculator"
    pm[f"{clc}{ro+1}"].font = xfont(bold=True, sz=9, color=HDR)
    pm[f"{clc}{ro+1}"].alignment = lft()
    pm.column_dimensions[clc].width  = 20
    pm.column_dimensions[cinp].width = 12

    # Max dose row
    pm[f"{clc}{ro+2}"] = "Max dose (µM)"
    pm[f"{clc}{ro+2}"].font = xfont(sz=9); pm[f"{clc}{ro+2}"].alignment = lft()
    inp_max = pm[f"{cinp}{ro+2}"]
    inp_max.value = round(concs[0], 4)
    inp_max.font  = xfont(bold=True, color="333333", sz=10)
    inp_max.fill  = PatternFill("solid", start_color="FFF9E6", fgColor="FFF9E6")
    inp_max.border = xborder(); inp_max.alignment = ctr()
    inp_max.number_format = "0.000"
    max_ref = f"{cinp}{ro+2}"   # absolute ref for formulas

    # Dilution factor row
    pm[f"{clc}{ro+3}"] = "Dilution factor"
    pm[f"{clc}{ro+3}"].font = xfont(sz=9); pm[f"{clc}{ro+3}"].alignment = lft()
    inp_dil = pm[f"{cinp}{ro+3}"]
    dil_val = w_dil.value if w_mode.value == "Auto-calculate" else (
              round(concs[0]/concs[1], 4) if len(concs) > 1 and concs[1] else 3.16)
    inp_dil.value = round(dil_val, 4)
    inp_dil.font  = xfont(bold=True, color="333333", sz=10)
    inp_dil.fill  = PatternFill("solid", start_color="FFF9E6", fgColor="FFF9E6")
    inp_dil.border = xborder(); inp_dil.alignment = ctr()
    inp_dil.number_format = "0.00"
    dil_ref = f"{cinp}{ro+3}"   # absolute ref for formulas

    pm[f"{clc}{ro+4}"] = "← edit above to recalculate"
    pm[f"{clc}{ro+4}"].font = xfont(italic=True, color="888888", sz=8)
    pm[f"{clc}{ro+4}"].alignment = lft()

    # ── Concentration label row ────────────────────────────────
    # Cells C-K get Excel formulas referencing the calculator inputs.
    # Cell C = max dose; each subsequent cell divides the previous by dil factor.
    # User can overwrite individual cells with plain values if needed.
    cr = ro + 10
    pm[f"A{cr}"].value = "µM →"; pm[f"A{cr}"].font = xfont(italic=True, sz=8); pm[f"A{cr}"].alignment = ctr()

    for di in range(DOSE_COUNT):
        ec = get_column_letter(di + 3)   # C, D, E ... K
        c  = pm[f"{ec}{cr}"]
        if di == 0:
            c.value = f"={max_ref}"
        else:
            prev_ec = get_column_letter(di + 2)
            c.value = f"={prev_ec}{cr}/{dil_ref}"
        c.font  = xfont(sz=8); c.fill = xfill(SUBHDR)
        c.alignment = ctr(); c.border = xborder()
        c.number_format = "0.000"

    c = pm[f"{get_column_letter(CTRL_COL+1)}{cr}"]
    c.value = "0 (ctrl)"; c.font = xfont(bold=True, sz=8)
    c.fill = xfill(CTRL); c.alignment = ctr(); c.border = xborder()

    # Legend — col O, dynamic CONCAT from Setup CONDITIONS table
    lc = get_column_letter(DOSE_COUNT+6)
    pm[f"{lc}{ro}"].value = f"Plate {plate_num}"
    pm[f"{lc}{ro}"].font  = xfont(bold=True)

    for ci in range(3):
        gc        = p*3 + ci + 1
        setup_row = COND_START + (gc - 1)
        color     = COND_PALETTE[(gc-1) % len(COND_PALETTE)]
        # Build CONCAT formula — Excel double-quotes are "" inside a string
        # =CONCAT("C1: ",B21," (",C21,") - ",D21," (",E21,")",IF(F21<>""," - "&F21,""))
        sr = setup_row
        # Build legend label using & operator (avoids @ implicit intersection)
        # Result: "C1: WT (WT-1) - Imatinib (imat-04)" or with additive appended
        sr = setup_row
        formula = (
            '="C' + str(gc) + ': "&Setup!B' + str(sr)
            + '&" ("&Setup!C' + str(sr)
            + '&") - "&Setup!D' + str(sr)
            + '&" ("&Setup!E' + str(sr)
            + '&")"&IF(Setup!F' + str(sr) + '<>""," - "&Setup!F' + str(sr) + ',"")'
        )
        c = pm[f"{lc}{ro+1+ci}"]
        c.value = formula
        c.fill  = xfill(color)
        c.font  = xfont(sz=9)
        c.border = xborder()
        c.alignment = lft()

    # Fixed entries: control and edge
    for li, (color, label) in enumerate([(CTRL, "Control (0 µM — col 11)"),
                                          (EDGE, "Edge / media wells")]):
        c = pm[f"{lc}{ro+4+li}"]
        c.value = label; c.fill = xfill(color)
        c.font  = xfont(sz=9); c.border = xborder(); c.alignment = lft()

# ── Save & download ──────────────────────────────────
fname = f"plate_layout_{exp_id}_{date_str}.xlsx".replace(" ", "_")
wb.save(fname)
files.download(fname)
print(f"✅  {fname}")
print(f"    {n_plates} plate(s)  ·  {n_conds} conditions  ·  {DOSE_COUNT} doses + control")
print()
print("Next steps:")
print("  1. Open the file and fill in Cell Line Lot + Compound Lot in the Setup sheet (columns B & C)")
print("  2. Run the assay using the Plate Maps sheet as reference")
print("  3. Upload plate_layout.xlsx + plate reader .xlsx to the analysis notebook")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅  plate_layout_test_2026-06-24.xlsx
    6 plate(s)  ·  18 conditions  ·  9 doses + control

Next steps:
  1. Open the file and fill in Cell Line Lot + Compound Lot in the Setup sheet (columns B & C)
  2. Run the assay using the Plate Maps sheet as reference
  3. Upload plate_layout.xlsx + plate reader .xlsx to the analysis notebook
